# ANR EEG Analysis Pipeline — Guided Google Colab

**African NeuroData Research Lab (ANR)**

This notebook guides you through an EEG analysis **one step at a time**. Run a cell, inspect the result, read the next explanation, then continue.

### Preferred ANR input
Upload the single **MNE-ready ZIP** generated by the ANR Muse EEG Recorder.

You may also upload the three extracted package files together:
- `*_raw_eeg.csv`
- `*_events.tsv`
- `*_eeg_metadata.json`

Legacy ANR CSV and common MNE-supported EEG formats remain available.

> **Research use only:** Technical QC and analysis outputs are not clinical EEG interpretation or diagnosis.

## Step 1 — Install the current ANR EEG Pipeline from GitHub

A fresh Colab runtime does not contain the ANR package. Run this cell first.

In [ ]:
import subprocess
import sys

REPO = "https://github.com/African-Neurodata-Research-Lab-ANR-LAB/ANR-EEG-Analysis-Pipeline.git"

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    f"git+{REPO}",
])

print("✓ ANR EEG Pipeline installed from GitHub")

## Step 2 — Import the analysis tools

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import anr_eeg

from anr_eeg import (
    load_eeg,
    run_qc,
    preprocess,
    compute_psd,
    compute_band_power,
    export_results,
    make_report,
)

print("✓ Imports successful")
print("ANR EEG version:", getattr(anr_eeg, "__version__", "unknown"))
print("MNE version:", mne.__version__)

## Step 3 — Upload your EEG recording

### Recommended
Upload **one `*_MNE_READY.zip` file** from the ANR Muse EEG Recorder.

### If the ZIP was already extracted
Select the matching raw EEG CSV, events TSV, and metadata JSON **together**. Colab places them in the same working directory and the ANR loader will associate them automatically.

### Other EEG formats
For EDF/BDF, FIF, BrainVision, or a legacy ANR CSV, upload the relevant recording file.

In [ ]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No EEG dataset was uploaded.")

uploaded_names = list(uploaded.keys())

zip_candidates = [
    name for name in uploaded_names
    if name.lower().endswith(".zip")
]
mne_raw_candidates = [
    name for name in uploaded_names
    if name.lower().endswith("_raw_eeg.csv")
]

if len(zip_candidates) == 1:
    file_path = zip_candidates[0]
    print("✓ Selected ANR MNE-ready ZIP:", file_path)
elif len(mne_raw_candidates) == 1:
    file_path = mne_raw_candidates[0]
    print("✓ Selected ANR MNE-ready raw EEG CSV:", file_path)
    print("  Companion metadata/events files will be detected automatically.")
elif len(uploaded_names) == 1:
    file_path = uploaded_names[0]
    print("✓ Selected EEG file:", file_path)
else:
    raise RuntimeError(
        "Upload one MNE-ready ZIP, or upload one matching "
        "*_raw_eeg.csv together with its metadata/events files."
    )

## Step 4 — Load the recording into MNE

For an ANR MNE-ready package, the loader validates the metadata, deterministic sample timing, channel structure, and events before creating an MNE `Raw` object.

For extracted ANR packages, `events.tsv` becomes the authoritative MNE annotation source.

In [ ]:
raw = load_eeg(file_path)

duration_s = raw.n_times / raw.info["sfreq"]

print(raw)
print()
print("Channels:", raw.ch_names)
print("Sampling frequency:", raw.info["sfreq"], "Hz")
print("Samples:", raw.n_times)
print("Sampled EEG duration:", round(duration_s, 6), "seconds")
print("Annotations/events:", len(raw.annotations))

try:
    source_info = json.loads(raw.info.get("description") or "{}")
except json.JSONDecodeError:
    source_info = {}

if source_info:
    print()
    print("ANR source information")
    print("----------------------")
    for key in [
        "source",
        "format_version",
        "timing_source",
        "sampling_rate_source",
        "session_code",
        "protocol",
        "wall_clock_duration_seconds",
        "eeg_duration_seconds",
    ]:
        if source_info.get(key) is not None:
            print(f"{key}: {source_info[key]}")

## Step 5 — Inspect a short segment of raw EEG

This plot is for research acquisition review, not diagnosis.

In [ ]:
PREVIEW_SECONDS = min(10.0, duration_s)

raw.plot(
    duration=PREVIEW_SECONDS,
    n_channels=min(4, len(raw.ch_names)),
    scalings="auto",
    show=False,
)
plt.show()

## Step 6 — Technical acquisition QC

The QC status describes recording/data quality only.

In [ ]:
qc = run_qc(raw)

print("Overall technical QC status:", qc["status"].upper())
print("Sampling frequency:", qc["sampling_frequency_hz"], "Hz")
print("Duration:", round(qc["duration_seconds"], 3), "seconds")
print("Channel count:", qc["channel_count"])
print("Annotations:", qc["annotation_count"])

qc_table = pd.DataFrame(qc["channels"]).T
display(qc_table)

## Step 7 — Choose preprocessing settings

Review these settings before continuing. Change them when your study design requires different parameters.

In [ ]:
LOW_CUT_HZ = 1.0
HIGH_CUT_HZ = 40.0
NOTCH_HZ = 50.0
REFERENCE = None

print("Band-pass:", LOW_CUT_HZ, "to", HIGH_CUT_HZ, "Hz")
print("Notch:", NOTCH_HZ, "Hz")
print("Reference:", REFERENCE)

## Step 8 — Preprocess

In [ ]:
clean = preprocess(
    raw,
    l_freq=LOW_CUT_HZ,
    h_freq=HIGH_CUT_HZ,
    notch=NOTCH_HZ,
    reference=REFERENCE,
)

print("✓ Preprocessing complete")
print(clean)

## Step 9 — Inspect the processed EEG

In [ ]:
clean.plot(
    duration=PREVIEW_SECONDS,
    n_channels=min(4, len(clean.ch_names)),
    scalings="auto",
    show=False,
)
plt.show()

## Step 10 — Power spectral density (PSD)

The PSD describes the distribution of signal power across frequency.

In [ ]:
spectrum = compute_psd(clean, fmin=1.0, fmax=40.0)
spectrum.plot(show=False)
plt.show()

## Step 11 — Relative EEG band power

The v1 core reports delta (1–4 Hz), theta (4–8 Hz), alpha (8–13 Hz), and beta (13–30 Hz), relative to total 1–30 Hz power per channel.

In [ ]:
bands = compute_band_power(clean)

band_percent = (bands * 100).round(2)
band_percent.columns = [
    f"{column}_pct" for column in band_percent.columns
]
display(band_percent)

## Step 12 — Visualize relative band power

In [ ]:
ax = band_percent.plot(kind="bar", figsize=(10, 5))
ax.set_title("ANR EEG Relative Band Power")
ax.set_xlabel("EEG channel")
ax.set_ylabel("Relative power (%)")
ax.legend(title="Frequency band")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 13 — Review event annotations

For MNE-ready ANR packages, these annotations come from `events.tsv`. Confirm that event timing and descriptions match your research protocol before event-related analyses.

In [ ]:
if len(clean.annotations) == 0:
    print("No event annotations were found in this recording.")
else:
    events_table = pd.DataFrame({
        "onset_seconds": clean.annotations.onset,
        "duration_seconds": clean.annotations.duration,
        "event": clean.annotations.description,
    })
    display(events_table)

## Step 14 — Generate standardized ANR outputs

Use a non-identifying session prefix.

In [ ]:
from pathlib import Path

SESSION_PREFIX = "anr_session"
output_dir = Path("/content/anr_eeg_results")

outputs = export_results(
    clean,
    qc,
    bands,
    output_dir,
    prefix=SESSION_PREFIX,
)

report_path = make_report(
    clean,
    qc,
    bands,
    output_dir / f"{SESSION_PREFIX}_report.html",
)

print("Generated outputs")
print("-----------------")
for name, path in outputs.items():
    print(f"{name}: {path}")
print("html_report:", report_path)

## Step 15 — Inspect generated files

In [ ]:
for path in sorted(output_dir.glob("*")):
    size_kb = path.stat().st_size / 1024
    print(f"{path.name:45s} {size_kb:10.1f} KB")

## Step 16 — Download the complete result package

Run this cell when you are satisfied with the analysis.

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(
    "/content/ANR_EEG_results",
    "zip",
    output_dir,
)

print("Downloading:", archive_path)
files.download(archive_path)

# Finished

**ANR MNE-ready recording → validated MNE Raw → technical QC → preprocessing → PSD → band power → event review → standardized outputs**

For input-format details, see `docs/MNE_READY_INPUT.md` in the repository.